# Evaluation

- Here, we calculate the **CER (Character Error Rate)** of the results from the fine-tuned SmolVLM model and comparing them with the Qwen2.5-VL 3B annotations.
- So, we are doing `CER(smolvlm_ft_data, qwen_annotation_data)`

In [1]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.1 MB/s eta 0:00:00


In [1]:
import glob
import jiwer
import torch
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForImageTextToText, AutoProcessor
from tqdm.auto import tqdm
from PIL import Image
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

## 1. Load data

### 1.1. Load Qwen Annotation data

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!unzip -q /content/drive/MyDrive/VLM/OCR-project.zip

In [10]:
qwen_annots_test_paths = glob.glob('./input/qwen2_5_vl_3b_annots/test_annots/*.txt')
qwen_annots_test_paths.sort()

qwen_annots_test_data = []
for file_path in qwen_annots_test_paths:
    data = open(file_path).read()
    qwen_annots_test_data.append(data.lower())
qwen_annots_test_data = qwen_annots_test_data[:12]
# example
print(qwen_annots_test_data[0])

tan chay yee
*** copy ***
ojc marketing sdn bhd
roc no: 538358-h
no 2 & 4, jalan bayu 4,
bandar seri alam,
81750 masai, johor
tel:07-388 2218 fax:07-388 8218
email: ng@ojcgroup.com
tax invoice
invoice no : pegiv-1030765
date : 15/01/2019 11:05:16 am
cashier : ng chuan min
sales persor : fatin
bill to : the peak quarry works
address :
description qty price amount
0000000111 1 193.00 193.00 sr
kings safety shoes kwd 805
qty: 1 total exclude gst: 193.00
total gst @6%: 0.00
total inclusive gst: 193.00
round amt: 0.00
total: 193.00
visa card 193.00
xxxxxx 4318
approval code:000
goods sold are not returnable & refundable
****thank you. please come again.****


### 1.2. Load SROIE dataset

In [11]:
all_image_paths = glob.glob('./input/sroie_v2/SROIE2019/test/img/*.jpg')
all_image_paths.sort()
all_image_paths = all_image_paths[:12]
print(len(all_image_paths))

12


### 1.3. Process dataset

In [12]:
class CustomData(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        return self.image_paths[idx]


dataset = CustomData(all_image_paths)
batched_dl = DataLoader(
    dataset=dataset,
    batch_size=4,
    shuffle=False,
    #num_workers=4
)
print(len(batched_dl))

3


### 1.2. Load Full fine-tuned model

In [5]:
#model_path = 'trained_models/full_ft/smolvlm2_256m_fullft_qwen2_5_vl_3b'
model_path = 'HuggingFaceTB/SmolVLM-256M-Instruct'
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    device_map='cuda',
    dtype=torch.bfloat16,
    _attn_implementation='eager', # Use `flash_attention_2` on Ampere GPUs and above and `eager` on older GPUs.
)

processor = AutoProcessor.from_pretrained(model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### 1.4. Inference on ft model

In [13]:
def generate_text_from_sample(model, processor, image_batch, message, max_new_tokens=1024, device='cuda'):
    # Prepare the text input
    messages = []
    for i, data in enumerate(image_batch):
      messages.append(message)
    text_input = processor.apply_chat_template(
        messages,  # Use the user message
        add_generation_prompt=True
    )

    # Display the text
    #print(text_input)

    # Prepare the image input
    image_inputs = []
    for data in image_batch:
      image = Image.open(data).convert('RGB')
      image_inputs.append(image)

    # Display the image
    # plt.figure(figsize=(8, 5))
    # plt.imshow(image_inputs[0])
    # plt.axis('off')
    # plt.show()

    # Prepare the inputs for the model
    model_inputs = processor(
        text=text_input,
        images=image_inputs,
        padding=True,
        padding_side='left',
        return_tensors='pt', # Return PyTorch tensors
    ).to(device, dtype=torch.bfloat16)

    # Generate text with the model
    generated_token_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    # Trim the generated token ids to remove the input token ids
    # Remove the original input tokens from the generated sequence
    trimmed_generated_token_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_token_ids)
    ]

    # Decode the output text
    # This converts the generated token IDs back into human-readable text
    output_text = processor.batch_decode(
        trimmed_generated_token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return output_text

In [14]:
inference_results = []
user_message = 'OCR this image accurately.'
message = [{'role': 'user','content': [
                {'type': 'image'},
                {'type': 'text', 'text': user_message}]},]

for i, batch in tqdm(enumerate(batched_dl), total=len(batched_dl)):
    outputs = generate_text_from_sample(model, processor,batch, message)
    # print(outputs)
    for output in outputs:
        inference_results.append(output.lower())

  0%|          | 0/3 [00:00<?, ?it/s]

In [15]:
print(len(inference_results))

12


## 3. Calculate CER

In [16]:
def calculate_cer(ground_truth, results):
    # Remove elements when ground truth has empty string elements.
    for i, gt in enumerate(ground_truth):
        if len(gt) == 0:
            ground_truth.pop(i)
            results.pop(i)

    error = jiwer.cer(ground_truth, results)
    print(f"CER: {error}")

In [17]:
calculate_cer(qwen_annots_test_data, inference_results)

CER: 2.4051802681244308
